# Proyecto universitario · Control inteligente y predicción de gastos personales

**Este cuaderno comienza desde cero.** No necesitas el Excel antiguo ni modelos previamente entrenados.

Objetivo: registrar ingresos y gastos, aplicar presupuestos por concepto y categoría, calcular alertas y saldos, dar consejos orientativos e investigar el desempeño de **Regresión Lineal, Random Forest y MLP** para pronosticar gastos.

El proyecto tiene **dos niveles**:

1. **Reglas exactas (sin IA):** alertas de presupuesto, saldos, totales y consejos basados en esos valores.
2. **Machine Learning (exploratorio):** estimaciones para el mes siguiente, comparadas con MAE, RMSE y R² mediante división temporal.

**Antes de ejecutar:** descarga y descomprime el archivo ZIP del proyecto. En este cuaderno cargarás primero el ZIP y después tu Excel diligenciado. Los datos de demostración son ficticios; no deben utilizarse en un informe como resultados reales.


## Etapa 1 · Instalar las bibliotecas

Colab ya incluye algunas bibliotecas; esta celda instala o actualiza las necesarias para el proyecto.


In [ ]:
!pip -q install "pandas>=2.1,<3" "openpyxl>=3.1,<4" "numpy>=1.26,<3" "scikit-learn>=1.4,<2" "joblib>=1.3,<2" "matplotlib>=3.7,<4"


## Etapa 2 · Cargar el paquete del proyecto

Selecciona el archivo **`Proyecto_Control_Gastos_IA_Completo.zip`** que descargaste de ChatGPT. Se descomprimirá en Colab y quedarán disponibles las funciones de validación, alertas y Machine Learning.


In [ ]:
from google.colab import files
from zipfile import ZipFile
import os, sys

subidos = files.upload()  # Seleccionar Proyecto_Control_Gastos_IA_Completo.zip
paquete = next((nombre for nombre in subidos if nombre.lower().endswith('.zip')), None)
if paquete is None:
    raise ValueError('Debes subir el ZIP completo del proyecto.')
with ZipFile(paquete) as z:
    z.extractall('/content')
ruta_proyecto = '/content/Proyecto_Control_Gastos_IA'
if not os.path.isdir(ruta_proyecto):
    raise FileNotFoundError('El ZIP debe contener la carpeta Proyecto_Control_Gastos_IA.')
os.chdir(ruta_proyecto)
sys.path.insert(0, ruta_proyecto)
print('Proyecto disponible en:', ruta_proyecto)


## Etapa 3 · Importar el código del proyecto

Todo el procesamiento se hará de forma **uniforme** para evitar errores como tomar SUELDO por gasto, sumar subtotales, o interpretar nombres de conceptos como encabezados.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from analisis import leer_excel, datos_demostracion, indicadores_mes, consejos_financieros, saldos_por_mes, detalle_por_concepto, detalle_por_categoria
from modelos import entrenar_modelos
print('Módulos cargados correctamente.')


## Etapa 4 · Subir tu Excel diligenciado

Llena en tu computadora la plantilla de la carpeta `plantilla`. Completa **MOVIMIENTOS** y **PRESUPUESTOS** con datos reales. La hoja **EJEMPLOS** no se utiliza en los análisis.

> Si todavía no tienes registros, pon `usar_ejemplo = True` para probar las etapas. Después cámbialo a `False` y carga tu Excel real.


In [ ]:
usar_ejemplo = False   # Cambia a True solo para probar con números FICTICIOS

if usar_ejemplo:
    mov, pre, config, avisos = datos_demostracion()
    print('MODO DEMOSTRACIÓN: los datos NO son reales.')
else:
    subidos_excel = files.upload()   # Seleccionar Plantilla_Control_Gastos_IA.xlsx (diligenciada)
    nombre_excel = next((nombre for nombre in subidos_excel if nombre.lower().endswith('.xlsx')), None)
    if nombre_excel is None:
        raise ValueError('Debes seleccionar un archivo .xlsx.')
    mov, pre, config, avisos = leer_excel(nombre_excel)

for aviso in avisos:
    print('AVISO:', aviso)
print('Filas de MOVIMIENTOS:', len(mov))
print('Filas de PRESUPUESTOS:', len(pre))
print('Configuración:', config)
display(mov.head(10))
display(pre.head(10))


## Etapa 5 · Revisión y preparación de los datos

Comprobamos meses disponibles, ingresos y gastos por separado, y categorías/ conceptos conocidos. Si la plantilla está incompleta, la función de carga señala las filas inválidas en lugar de producir una suma errónea.


In [ ]:
print('Meses con movimientos:', sorted(mov['Mes'].unique()) if not mov.empty else [])
print('Categorías con gastos:', sorted(mov.loc[mov['Tipo']=='Gasto','Categoria'].unique()) if not mov.empty else [])
print('Total ingresos:', mov.loc[mov['Tipo']=='Ingreso','Valor_COP'].sum() if not mov.empty else 0)
print('Total gastos:', mov.loc[mov['Tipo']=='Gasto','Valor_COP'].sum() if not mov.empty else 0)
print('Saldo inicial:', config['saldo_inicial'])

detalle = detalle_por_concepto(mov, pre, config['umbral'])
por_categoria = detalle_por_categoria(detalle, config['umbral'])
display(detalle.head(20))


## Etapa 6 · Calcular el saldo y las alertas

Las alertas se basan en la razón **gasto real / presupuesto**. Sin presupuesto, cualquier gasto registrado se señala como **SIN PRESUPUESTO**. En el umbral aparece **ADVERTENCIA** y al superar el 100 % aparece **EXCEDIDO**.


In [ ]:
meses = sorted(set(mov['Mes'].tolist()) | set(pre['Mes'].tolist()))
if not meses:
    raise ValueError('La plantilla todavía no contiene movimientos ni presupuestos.')
mes_analisis = config['mes'] if config['mes'] in meses else meses[-1]
resultado_mes = indicadores_mes(mov, pre, mes_analisis, config['saldo_inicial'], config['umbral'])
print('Mes analizado:',mes_analisis)
for nombre in ['Ingresos_COP','Gastos_COP','Balance_COP','Saldo_COP','Presupuesto_COP','Disponible_Presupuesto_COP']:
    print(f'{nombre}: ${resultado_mes[nombre]:,.0f} COP')
print('\nControl por conceptos:')
display(resultado_mes['conceptos'])
print('\nControl por categorías:')
display(resultado_mes['categorias'])
print('\nConsejos basados en saldos reales:')
for nivel, texto in consejos_financieros(resultado_mes):
    print(nivel, '·', texto)


## Etapa 7 · Análisis exploratorio y gráficas

Estas gráficas explican el histórico y muestran en qué categorías hubo exceso. Los importes son directamente los de tus datos validados.


In [ ]:
historico = saldos_por_mes(mov, config['saldo_inicial'])
if not historico.empty:
    ax = historico.plot(x='Mes',y=['Ingresos_COP','Gastos_COP','Saldo_COP'],marker='o',figsize=(11,4),title='Evolución mensual')
    ax.set_ylabel('COP')
    ax.grid(True,alpha=0.2)
    plt.xticks(rotation=45)
    plt.tight_layout();plt.show()

cats = resultado_mes['categorias']
if not cats.empty:
    ax = cats.set_index('Categoria')[['Presupuesto_COP','Real_COP']].plot.bar(figsize=(12,4),title=f'Presupuesto y gasto por categoría · {mes_analisis}')
    ax.set_ylabel('COP');plt.xticks(rotation=45,ha='right');plt.tight_layout();plt.show()


## Etapa 8 · Entrenar y evaluar Regresión Lineal, Random Forest y MLP

**Sin fuga temporal de información:** las variables para pronosticar cada concepto utilizan su presupuesto y los gastos de los **dos meses anteriores**. Reservamos los dos meses evaluables más recientes para prueba. Se comparan **MAE, RMSE y R²**. Con pocos meses o datos incompletos, el entrenamiento se bloquea e informa el motivo.

La predicción es orientativa y puede fallar. No debe presentarse como garantía de un resultado futuro.


In [ ]:
resultados_ml = None
try:
    resultados_ml = entrenar_modelos(mov, pre, config['umbral'])
    print('Modelo con menor MAE:', resultados_ml['mejor_modelo'])
    print('Meses reservados para prueba:', resultados_ml['meses_test'])
    print(resultados_ml['advertencia'])
    display(resultados_ml['metricas'])
    resultados_ml['metricas'].plot.bar(x='Modelo',y='MAE_COP',rot=0,figsize=(9,4),title='Error absoluto medio (MAE) en datos de prueba')
    plt.tight_layout();plt.show()
except ValueError as err:
    print('Aún no es posible entrenar:', err)
    print('Las alertas y consejos anteriores sí están disponibles con los datos actuales.')


## Etapa 9 · Pronóstico del siguiente mes y riesgo estimado

Si existe un presupuesto futuro en **PRESUPUESTOS**, el modelo lo utiliza. Si no existe, se utiliza el último presupuesto como **provisional** y se marca explícitamente. El modelo no usa los gastos reales del mes que intenta predecir.


In [ ]:
if resultados_ml is not None:
    predicciones = resultados_ml['pronostico']
    display(predicciones)
    pronostico_categorias = predicciones.groupby(['Mes','Categoria'],as_index=False)[['Presupuesto_COP','Prediccion_COP']].sum()
    pronostico_categorias['Exceso_estimado_COP'] = (pronostico_categorias['Prediccion_COP']-pronostico_categorias['Presupuesto_COP']).clip(lower=0)
    print('Pronóstico total por categoría:')
    display(pronostico_categorias)
    print('Total proyectado:', f'${predicciones["Prediccion_COP"].sum():,.0f} COP')
    print('Conceptos con riesgo predicho de excederse:', (predicciones['Riesgo_predicho']=='EXCEDIDO').sum())
else:
    print('Completa al menos 8 meses con registros, preferiblemente 12 o más meses completos.')


## Etapa 10 · Exportar resultados y modelos

Descarga un ZIP con los CSV de análisis. Cuando el entrenamiento haya sido posible, se incluyen las métricas, el pronóstico y un archivo `.joblib` con el modelo ganador. **No cargues modelos `.joblib` de fuentes desconocidas**, porque usan serialización de Python.


In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
import joblib

salida = Path('/content/resultados_control_gastos')
salida.mkdir(exist_ok=True)
resultado_mes['conceptos'].to_csv(salida/'control_conceptos.csv',index=False,encoding='utf-8-sig')
resultado_mes['categorias'].to_csv(salida/'control_categorias.csv',index=False,encoding='utf-8-sig')
historico.to_csv(salida/'saldos_mensuales.csv',index=False,encoding='utf-8-sig')
if resultados_ml is not None:
    resultados_ml['metricas'].to_csv(salida/'metricas_modelos.csv',index=False,encoding='utf-8-sig')
    resultados_ml['pronostico'].to_csv(salida/'pronostico.csv',index=False,encoding='utf-8-sig')
    joblib.dump(resultados_ml['modelos'][resultados_ml['mejor_modelo']],salida/'modelo_ganador.joblib')
archivo_zip='/content/resultados_control_gastos.zip'
with ZipFile(archivo_zip,'w',ZIP_DEFLATED) as z:
    for archivo in salida.glob('*'):
        z.write(archivo,arcname=archivo.name)
files.download(archivo_zip)


## Etapa 11 · Aplicación Streamlit (en tu computador)

Al terminar el análisis en Colab, puedes ejecutar la **aplicación completa** con el código incluido en el ZIP:

```bash
cd Proyecto_Control_Gastos_IA
python -m venv .venv
# Windows PowerShell: .venv\Scripts\Activate.ps1
# Windows Git Bash: source .venv/Scripts/activate
pip install -r requirements.txt
streamlit run app.py
```

La aplicación permite descargar la plantilla, subir datos llenos, visualizar control por concepto y categoría, obtener consejos basados en saldo y entrenar los mismos modelos. **No necesitas copiar las predicciones de Colab a la aplicación**: ambas usan el mismo código fuente `analisis.py` y `modelos.py`.

Cuando tengas resultados reales, documenta tamaño de muestra, periodos, origen de datos y limitaciones del ML.
